# 5. Expert Parallelism

Expert parallelism is commonly used in **Mixture-of-Experts (MoE)**
models.

**Core idea**
- the model has several experts
- a router decides which expert should process each token
- different experts can live on different GPUs
- only a subset of experts runs for each token

This is the most advanced notebook in the set because it adds
**conditional computation** on top of parallelism.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Beginner mental model

Imagine a team with specialists:
- one expert is good at one type of token
- another expert is good at another type
- the router sends each token to one specialist

The big win is that the whole model can have many experts, while a
single token only uses a small part of them.

In [ ]:
class Expert(nn.Module):
    """A tiny expert network."""

    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Top1MoE(nn.Module):
    """Routes each token to exactly one expert."""

    def __init__(self, hidden_size: int, num_experts: int) -> None:
        super().__init__()
        self.router = nn.Linear(hidden_size, num_experts, bias=False)
        self.experts = nn.ModuleList([Expert(hidden_size) for _ in range(num_experts)])

    def forward(
        self,
        tokens: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, dict[int, int]]:
        router_scores = self.router(tokens)
        chosen_experts = router_scores.argmax(dim=-1)
        outputs = torch.zeros_like(tokens)
        load_per_expert: dict[int, int] = {}

        for expert_id, expert in enumerate(self.experts):
            mask = chosen_experts == expert_id
            load_per_expert[expert_id] = int(mask.sum().item())
            if mask.any():
                outputs[mask] = expert(tokens[mask])

        return outputs, chosen_experts, load_per_expert


num_tokens = 8
hidden_size = 4
num_experts = 3

tokens = torch.randn(num_tokens, hidden_size)
moe = Top1MoE(hidden_size, num_experts)

expert_outputs, expert_ids, load_per_expert = moe(tokens)
final_output = tokens + 0.3 * expert_outputs

print("Token shape:", tuple(tokens.shape))
print("Chosen expert per token:", expert_ids.tolist())
print("Load per expert:", load_per_expert)
print("Final output shape:", tuple(final_output.shape))

In [ ]:
# Show exactly which tokens were sent to each expert.
for expert_id in range(num_experts):
    token_positions = (expert_ids == expert_id).nonzero(as_tuple=False).squeeze(-1).tolist()
    print(f"Expert {expert_id} receives token positions: {token_positions}")

assert expert_outputs.shape == tokens.shape
assert final_output.shape == tokens.shape
assert sum(load_per_expert.values()) == num_tokens

print()
print("Every token was routed to one expert, and the output shape is correct.")

## Takeaways

- Expert parallelism distributes **experts**, not just tensors or
  batches.
- The router is responsible for sending tokens to experts.
- MoE models can increase model capacity without forcing every token
  to use the full network.
- Real systems also worry about load balancing so one expert does
  not receive almost everything.